# Telegram monitor one-time setup
이 노트북은 텔레그램 세션 문자열과 알림 받을 chat ID를 한 번 생성합니다. 출력값을 GitHub Secrets로 옮긴 뒤 노트북 출력과 런타임을 삭제하세요. 누구에게도 출력값을 공유하지 마세요.

In [ ]:
!pip -q install "Telethon>=1.40,<2"

import json
import urllib.request
from getpass import getpass
from telethon import TelegramClient
from telethon.errors import SessionPasswordNeededError
from telethon.sessions import StringSession

api_id = int(input("TELEGRAM_API_ID: "))
api_hash = getpass("TELEGRAM_API_HASH: ")
phone = input("전화번호(국가번호 포함, 예: +821012345678): ").strip()

client = TelegramClient(StringSession(), api_id, api_hash)
await client.connect()
await client.send_code_request(phone)
code = input("텔레그램으로 받은 로그인 코드: ").strip()
try:
    await client.sign_in(phone, code)
except SessionPasswordNeededError:
    await client.sign_in(password=getpass("텔레그램 2단계 인증 비밀번호: "))
session_string = client.session.save()
await client.disconnect()

print("\nTELEGRAM_SESSION (전체를 복사):\n", session_string)
print("\n이제 아이폰 텔레그램에서 알림 봇을 열고 /start를 보내세요.")
bot_token = getpass("BOT_TOKEN: ").strip()
input("/start를 보냈다면 Enter: ")
url = f"https://api.telegram.org/bot{bot_token}/getUpdates"
with urllib.request.urlopen(url, timeout=30) as response:
    updates = json.loads(response.read().decode("utf-8"))
private_chats = [
    item["message"]["chat"]["id"]
    for item in updates.get("result", [])
    if item.get("message", {}).get("chat", {}).get("type") == "private"
]
if private_chats:
    print("\nALERT_CHAT_ID:", private_chats[-1])
else:
    print("\nchat ID를 찾지 못했습니다. 봇에 /start를 보낸 뒤 이 셀을 다시 실행하세요.")
